In [0]:
from pyspark.sql.functions import col, count, when,desc

apps_df = spark.read.table("silver_applications")
courses_df = spark.read.table("silver_courses")
uni_df = spark.read.table("silver_universities")
students_df = spark.read.table("silver_students")

#Business requirement: Student Nationality Analysis
nationality_summary_df = (
    spark.read.table("silver_students")
    .groupBy(col("nationality"))
    .agg(count("student_id").alias("total_students"))
    .orderBy(desc("total_students"))
)
#How many students belong to each education level, and which education level has the highest number of students?
education_summary_df = (
    spark.read.table("silver_students")
    .groupBy("education_level")
    .agg(count("student_id").alias("total_students"))
    .orderBy(desc("total_students"))
)
#How many applications has each student submitted?


In [0]:
students_df = spark.read.table(
    "workspace.default.silver_students"
)

students_df.show()

In [0]:
from pyspark.sql.functions import desc,col,count

students_df = spark.read.table(
    "workspace.default.silver_students"
)

nationality_summary_df = (
    students_df
    .groupBy("nationality","education_level")
    .agg(
        count("student_id").alias("total_students")
    )
    .orderBy(desc("total_students"))
)

nationality_summary_df.show(truncate=False)

In [0]:
(
    nationality_summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.education_lakehouse.gold_student_nationality_summary"
    )
)

gold_df = spark.read.table("workspace.education_lakehouse.gold_student_nationality_summary")
gold_df.show(truncate=False)
print("Total records:", gold_df.count())

In [0]:

from pyspark.sql.functions import count, desc

# Step 1: Read Silver data and aggregate
application_status_summary_df = (
    spark.read.table(
        "workspace.default.silver_applications"
    )
    .groupBy("application_status")
    .agg(
        count("application_id").alias("total_applications")
    )
    .orderBy(desc("total_applications"))
)

# Step 2: Display aggregation
application_status_summary_df.show(truncate=False)

# Step 3: Save as Gold Delta table
(
    application_status_summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.education_lakehouse.gold_application_status_summary"
    )
)

# Step 4: Read and validate
gold_applications_df = spark.read.table(
    "workspace.education_lakehouse.gold_application_status_summary"
)

gold_applications_df.show(truncate=False)

print("Total records:", gold_applications_df.count())


In [0]:
applications_df = spark.read.table("workspace.default.silver_applications")
students_df = spark.read.table("workspace.default.silver_students")
courses_df = spark.read.table("workspace.default.silver_courses")
universities_df = spark.read.table("workspace.default.silver_universities")

applications_df.printSchema()
students_df.printSchema()
courses_df.printSchema()
universities_df.printSchema()

In [0]:
from pyspark.sql.functions import col

application_student_df = (
    applications_df.alias("a")
    .join(
        students_df.alias("s"),
        col("a.student_id") == col("s.student_id"),
        "left"
    )
)
print("Applications:", applications_df.count())
print("Joined records:", application_student_df.count())

unmatched_students_df = application_student_df.filter(
    col("s.student_id").isNull()
)
print("Applications without matching students",unmatched_students_df.count())

In [0]:
from pyspark.sql.functions import col
#Joing Applications with Courses dataset
application_student_course_df = (
    application_student_df.alias("ac")
    .join(
        courses_df.alias("c"),
        col("ac.course_id") == col("c.course_id"),
        "left"
    )
)
print("Previous Records",applications_df.count())
print("After Joining Records",application_student_course_df.count())

unmatched_app_df = application_student_course_df.filter(
    col("c.course_id").isNull()
)
print("Applications without matching course_id",unmatched_app_df.count())

In [0]:
from pyspark.sql.functions import col

application_student_course_uni_df = (
    application_student_course_df.alias("acs")
    .join(
        universities_df.alias("u"),
        col("acs.university_id") == col("u.university_id"),
        "left" 
    )
)
print("Previous Records:", application_student_course_df.count())
print("After Joining Records:", application_student_course_uni_df.count())

unmatched_uni_df = application_student_course_uni_df.filter(
    col("u.university_id").isNull()
)
print("Universities without matching uni_id:", unmatched_uni_df.count())

In [0]:
from pyspark.sql.functions import count

duplicate_applications_df = (
    application_student_course_uni_df
    .groupBy("acs.application_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

print("Duplicate application IDs:", duplicate_applications_df.count())

duplicate_applications_df.show()